# 03 · Inference & Execution-Accuracy Evaluation

Generate SQL on BIRD dev with the fine-tuned adapter, then score with
execution accuracy (EX), valid-SQL rate and exact match — plus a baseline
(same model, no fine-tuning) for a clean before/after comparison.

**Self-sufficient:** runs in a *fresh* session — it pulls the fine-tuned
adapter from the **Hugging Face Hub** and rebuilds the dev set if needed. (If
you're in the same session as notebook 02, it reuses what's already there.)

In [ ]:
# --- SETUP: run me first (works on Colab, Kaggle AND locally) ---
import os, sys, subprocess
if not os.path.exists('src') and os.path.basename(os.getcwd()) != 'text2sql-finetuning':
    if not os.path.isdir('text2sql-finetuning'):
        subprocess.run(['git', 'clone', 'https://github.com/Shiverion/text2sql-finetuning.git'], check=True)
    os.chdir('text2sql-finetuning')
sys.path.insert(0, os.getcwd())
print('working dir :', os.getcwd())
print('src present :', os.path.isdir('src'))

In [ ]:
!pip install -q unsloth   # needed for inference if this is a fresh session

## 1. Choose the fine-tuned model
Defaults to your HF adapter repo (works in any session). If you're in the same
session as notebook 02, you can point this at the local `outputs/...` folder.

In [ ]:
FT_MODEL = 'Shiverion/qwen2.5-coder-1.5b-bird-qlora'   # <- your HF repo from notebook 02
# FT_MODEL = 'outputs/qwen2.5-coder-1.5b-bird-qlora'   # use this if same session as nb 02
BASE_MODEL = 'unsloth/Qwen2.5-Coder-1.5B-Instruct'
print('fine-tuned :', FT_MODEL)

## 2. Prepare the dev set (idempotent)
Downloads + preprocesses BIRD dev only if `bird_dev.jsonl` isn't already here.
The real `.sqlite` databases are required to *execute* the predicted queries.

In [ ]:
import os, glob
REPO = os.getcwd(); BIRD = 'https://bird-bench.oss-cn-beijing.aliyuncs.com'
dev_jsonl = 'data/processed/bird_dev.jsonl'
if os.path.exists(dev_jsonl) and os.path.getsize(dev_jsonl) > 0:
    print('dev set already prepared:', dev_jsonl)
else:
    if not glob.glob('dev*/dev.json'):
        print('downloading dev.zip (~330 MB)...', flush=True)
        !wget --progress=dot:giga -c {BIRD}/dev.zip -O /tmp/dev.zip
        !unzip -o -q /tmp/dev.zip -d {REPO}
        !rm -f /tmp/dev.zip
    for z in glob.glob('dev*/dev_databases.zip'):
        print('un-nesting', z, '...', flush=True)
        !unzip -o -q '{z}' -d '{os.path.dirname(z)}'
        !rm -f '{z}'
    dj = glob.glob('dev*/dev.json')[0]; dd = os.path.dirname(dj) + '/dev_databases'
    !python -m src.data_prep --source bird --json {dj} --db_root {dd} --out {dev_jsonl}
n = sum(1 for _ in open(dev_jsonl, encoding='utf-8'))
print(f'{dev_jsonl}: {n} records (must be > 0 for execution eval)')

## 3. Baseline — base model, zero-shot
Establishes the lift attributable to fine-tuning. (`--limit 200` keeps it
quick; remove for the full dev set.)

In [ ]:
!python -m src.inference --model_dir {BASE_MODEL} \
    --input data/processed/bird_dev.jsonl --output outputs/preds_base.jsonl --limit 200
!python -m src.evaluate --pred outputs/preds_base.jsonl --report outputs/metrics_base.json

## 4. Fine-tuned model (from Hugging Face)
`src.inference` loads the adapter and its base model automatically (Unsloth
reads `base_model_name_or_path` from the adapter config).

In [ ]:
!python -m src.inference --model_dir {FT_MODEL} \
    --input data/processed/bird_dev.jsonl --output outputs/preds_ft.jsonl --limit 200
!python -m src.evaluate --pred outputs/preds_ft.jsonl --report outputs/metrics_ft.json

## 5. Compare

In [ ]:
import json, pandas as pd, os
os.makedirs('report/figures', exist_ok=True)
base = json.load(open('outputs/metrics_base.json'))
ft   = json.load(open('outputs/metrics_ft.json'))
tbl = pd.DataFrame({'baseline': base, 'fine-tuned': ft}).loc[
    ['execution_accuracy','valid_sql_rate','exact_match']]
print(tbl)
ax = tbl.plot(kind='bar', rot=0, title='Baseline vs fine-tuned'); ax.figure.tight_layout()
ax.figure.savefig('report/figures/before_after.png', dpi=120)

## 6. Error analysis
Read the captured failures to find systematic mistakes (hallucinated columns,
wrong joins, missing GROUP BY). This is what drives the next experiment.

In [ ]:
for e in ft.get('error_samples', [])[:10]:
    print(e['type'])
    print('  pred:', e.get('pred',''))
    if 'error' in e: print('  err :', e['error'])
    print()

## 7. Try the motivating question live
Uses the bundled fintech DB, so it works even without BIRD downloaded.

In [ ]:
from src.inference import load_model, generate_batch
from src.prompts import build_messages, extract_sql
from src.schema_utils import serialize_schema
model, tok, _ = load_model(FT_MODEL, 2048, load_in_4bit=True)
tok.padding_side='left'; tok.pad_token = tok.pad_token or tok.eos_token
schema = serialize_schema('data/sample/db/fintech/fintech.sqlite')
q = 'Who were the top performing merchants last quarter?'
prompt = tok.apply_chat_template(build_messages(schema, q), tokenize=False, add_generation_prompt=True)
print(extract_sql(generate_batch(model, tok, [prompt], 256)[0]))